In [14]:
import pandas as pd
import re
from difflib import SequenceMatcher

# -----------------------------
# 1. 데이터 로드
# -----------------------------
map_df = pd.read_csv("서울시_행정동_법정동_맵핑.csv", encoding="cp949")
id_df = pd.read_csv("서울_행정동ID.csv", encoding="utf-8-sig")

# -----------------------------
# 2. manual_map 정의
# -----------------------------
manual_map = {
    "청운효자동": ["청운동", "효자동"],
    "종로1234가동": ["종로1가", "종로2가", "종로3가", "종로4가"]
}

# -----------------------------
# 3. 문자열 정제 함수
# -----------------------------
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[^\w가-힣]", "", text)
    return text

# -----------------------------
# 4. manual_map 확장 함수
# -----------------------------
def expand_name(name):
    clean = clean_text(name)

    if clean in manual_map:
        # 확장된 후보 리스트 반환
        return manual_map[clean]

    return [name]

# -----------------------------
# 5. 유사도 함수
# -----------------------------
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# -----------------------------
# 6. 전처리
# -----------------------------
map_df["clean_name"] = map_df["행정동이름"].apply(clean_text)

# -----------------------------
# 7. 매핑 수행
# -----------------------------
results = []

for _, row in id_df.iterrows():
    best_score = -1
    best_match = None

    # manual_map 적용된 후보 리스트
    candidates = expand_name(row["행정동_명칭"])

    for candidate in candidates:
        clean_candidate = clean_text(candidate)

        for _, mrow in map_df.iterrows():
            score = similarity(clean_candidate, mrow["clean_name"])

            if score > best_score:
                best_score = score
                best_match = mrow

    results.append({
        "행정동_ID": row["행정동_ID"],
        "행정동_명칭": row["행정동_명칭"],
        "매핑_행정동코드": best_match["행정동코드"],
        "매핑_행정동이름": best_match["행정동이름"],
        "유사도": round(best_score, 4)
    })

res_df = pd.DataFrame(results)

# -----------------------------
# 8. 결과 저장
# -----------------------------
res_df.to_csv("행정동ID_매핑결과.csv", index=False, encoding="utf-8-sig")

# -----------------------------
# 9. 품질 체크
# -----------------------------
print("=== 유사도 통계 ===")
print(res_df["유사도"].describe())

print("\n=== 낮은 유사도 (<0.85) ===")
print(res_df[res_df["유사도"] < 0.85])

=== 유사도 통계 ===
count    426.000000
mean       0.950256
std        0.060128
min        0.666700
25%        0.888900
50%        1.000000
75%        1.000000
max        1.000000
Name: 유사도, dtype: float64

=== 낮은 유사도 (<0.85) ===
       행정동_ID       행정동_명칭    매핑_행정동코드     매핑_행정동이름     유사도
7    11010610  종로1.2.3.4가동  1111061500  종로1.2.3.4가동  0.6667
15   11010720        청운효자동  1111051500        청운효자동  0.7500
275  11170740           항동  1150062000          공항동  0.8000
359  11230511         개포3동  1165058000         반포3동  0.7500
419  11250710         둔촌2동  1117064000        이촌제2동  0.6667


In [15]:
print(len(res_df['행정동_ID'].unique()))
print(len(res_df['매핑_행정동코드'].unique()))



426
421


In [3]:
import pandas as pd


# -----------------------------
# 1. 데이터 로드
# -----------------------------
code_emd_df = pd.read_csv("서울시_행정동_법정동_맵핑.csv", encoding="cp949")
id_code_df = pd.read_csv("서울시_행정동ID_행정동코드_맵핑.csv", encoding="cp949")

mapping_df = pd.merge(
    id_code_df,
    code_emd_df,
    on="행정동코드",
    how="outer"
)

# -----------------------------
# 8. 결과 저장
# -----------------------------
mapping_df.to_csv("서울시_행정동ID_행정동코드_법정동코드_매핑.csv",
              index=False, encoding="utf-8-sig")
